<a href="https://colab.research.google.com/github/Albert1616/ProjetoCAD-KMeans/blob/KmeansCUDA/KmeansCUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile aluno.h

#ifndef ALUNO_H
#define ALUNO_H

typedef struct
{
    float media;
    float numeroFaltas;
    int cluster;
} Aluno;

#endif // ALUNO_H

In [ ]:
%%writefile dataset.h

#include "aluno.h"
#ifndef DATASET_H
#define DATASET_H

__host__ char *retornarDadoPorIndice(char *linha, int index);
__host__ int carregarDataset(Aluno *alunos);
__host__ void normalizarAlunos(Aluno *alunos, int total);

#endif // DATASET_H

In [ ]:
%%writefile kmeans.h

#ifndef KMEANS_H
#define KMEANS_H
#include "Aluno.h"

typedef struct
{
    int k;
    int max_iter;
    int random_state;
    int totalAlunos;
    Aluno *centroids;
} KMeans;

__device__ void initCentroids(KMeans *model, Aluno *alunos);
__device__ void assignClusters(KMeans *model, Aluno *alunos);
__device__ void updateCentroids(KMeans *model, Aluno *alunos);
__device__ void predict(KMeans *model, Aluno *novoAluno);
__device__ void fit(KMeans *model, Aluno *alunos);
__device__ float *methodElbow(KMeans *model, Aluno *alunos);

#endif // KMEANS_H



In [ ]:
%%writefile kmeans.cu
#include <math.h>
#include <stdlib.h>
#include "aluno.h"
#include "kmeans.h"

// Utilitários para o KMeans
__device__ float distEuclidiana(Aluno *aluno, Aluno *centroid)
{
    float diffMedia = pow(aluno->media - centroid->media, 2);
    float diffFaltas = pow(aluno->numeroFaltas - centroid->numeroFaltas, 2);

    return sqrt(diffMedia + diffFaltas);
}

__device__ int minListaIndex(float lista[], int tamanho)
{
    float min = lista[0];
    int index = 0;

    for (int i = 0; i < tamanho; i++)
    {
        if (lista[i] < min)
        {
            index = i;
            min = lista[i];
        }
    }

    return index;
}

__device__ void initCentroids(KMeans *model, Aluno *alunos)
{
    int listIndex[model->k];
    for (int i = 0; i < model->k; i++)
        listIndex[i] = -1;

    srand(model->random_state);

    for (int i = 0; i < model->k; i++)
    {
        int index = rand() % model->totalAlunos;
        int duplicado = 0;

        for (int j = 0; j < i; j++)
        {
            if (listIndex[j] == index)
            {
                duplicado = 1;
                break;
            }
        }

        if (duplicado)
        {
            i--;
            continue;
        }

        listIndex[i] = index;
        model->centroids[i] = alunos[index];
    }
}

__device__ void assignClusters(KMeans *model, Aluno *alunos, int i)
{
    float distanciaClusters[model->k];

    for (int i = 0; i < model->totalAlunos; i++)
    {
        for (int j = 0; j < model->k; j++)
        {
            float distancia = distEuclidiana(&alunos[i], &model->centroids[j]);
            distanciaClusters[j] = distancia;
        }

        alunos[i].cluster = minListaIndex(distanciaClusters, model->k);
    }
}

__device__ void updateCentroids(KMeans *model, Aluno *alunos)
{
    Aluno newCentroids[model->k];
    int countAlunos[model->k];

    for (int i = 0; i < model->k; i++)
    {
        newCentroids[i].media = 0;
        newCentroids[i].numeroFaltas = 0;
        countAlunos[i] = 0;
    }

    for (int i = 0; i < model->totalAlunos; i++)
    {
        int clusterIndex = alunos[i].cluster;

        newCentroids[clusterIndex].media += alunos[i].media;
        newCentroids[clusterIndex].numeroFaltas += alunos[i].numeroFaltas;
        countAlunos[clusterIndex]++;
    }

    for (int i = 0; i < model->k; i++)
    {
        if (countAlunos[i] > 0)
        {
            model->centroids[i].media = newCentroids[i].media / countAlunos[i];
            model->centroids[i].numeroFaltas = newCentroids[i].numeroFaltas / countAlunos[i];
        }
    }
}

__global__ void fit(KMeans *model, Aluno *alunos)
{
  int i = blockIdx.x * blockDim.x + threadIdx.x;

  if (i == 0)
    initCentroids(model, alunos);

  __syncthreads();

  if (i < model->totalAlunos)
  {
    for (int i = 0; i < model->max_iter; i++)
    {
        Aluno old_centroids[model->k];
        for (int j = 0; j < model->k; j++)
            old_centroids[j] = model->centroids[j];

        assignClusters(model, alunos, i);
        __syncthreads();

        if (i == 0)
          updateCentroids(model, alunos);

        __syncthreads()

        int convergiu = 1;

        for (int j = 0; j < model->k; j++)
        {
            if (distEuclidiana(&model->centroids[j], &old_centroids[j]) > 0.01)
            {
                convergiu = 0;
                break;
            }
        }

        if (convergiu)
        {
            break;
        }
    }
  }
}

__global__ void predict(KMeans *model, Aluno *novoAluno)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < model->totalAlunos){
      float distanciaClusters[model->k];

      for (int i = 0; i < model->k; i++)
      {
          distanciaClusters[i] = distEuclidiana(novoAluno, &model->centroids[i]);
      }

      novoAluno->cluster = minListaIndex(distanciaClusters, model->k);
    }
}

__device__ float *methodElbow(KMeans *model, Aluno *alunos)
{
    int k_range[] = {2, 3, 4, 5, 6, 7, 8};
    int n = 7;
    float *inertia_values = (float *)malloc(n * sizeof(float));

    for (int i = 0; i < n; i++)
    {
        model->k = k_range[i];
        free(model->centroids);
        model->centroids = (Aluno *)malloc(model->k * sizeof(Aluno));

        fit(model, alunos);

        float inertia = 0;
        for (int j = 0; j < model->totalAlunos; j++)
            inertia += pow(distEuclidiana(&alunos[j], &model->centroids[alunos[j].cluster]), 2);

        inertia_values[i] = inertia;
    }
    return inertia_values;
}


In [2]:
%%writefile KmeansCUDA.cu

#include <cuda_runtime_api.h>
#include <memory.h>
#include <cstdlib>
#include <ctime>
#include <stdio.h>
#include <cuda/cmath>
#include <chrono>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define NUM_CLUSTERS 3
#define MAX_ITERATIONS 10000

__host__ void exportarResultados(Aluno *alunos, int total)
{
    FILE *file = fopen("resultados.csv", "w");
    fprintf(file, "numeroFaltas,media,cluster\n");

    for (int i = 0; i < total; i++)
        fprintf(file, "%.2f,%.2f,%d\n",
                alunos[i].numeroFaltas,
                alunos[i].media,
                alunos[i].cluster);

    fclose(file);
}

int main()
{
    // Dados de cada aluno - CPU
    Aluno *h_alunos = (Aluno *)malloc(3000 * sizeof(Aluno));
    int numeroAlunos = carregarDataset(h_alunos);

    normalizarAlunos(h_alunos, numeroAlunos);

    Aluno *d_alunos;
    cudaMalloc(&d_alunos, numeroAlunos * sizeof(Aluno));
    cudaMemcpy(d_alunos, alunos, numeroAlunos * sizeof(Aluno), cudaMemcpyHostToDevice);

    // // Configuração do modelo
    KMeans h_kmeans;
    h_kmeans.k = NUM_CLUSTERS;
    h_kmeans.max_iter = MAX_ITERATIONS;
    h_kmeans.random_state = 42;
    h_kmeans.centroids = (Aluno *)malloc(h_kmeans.k * sizeof(Aluno));
    h_kmeans.totalAlunos = numeroAlunos;

    Kmeans d_kmeans;
    cudaMalloc(&d_kmeans, sizeof(Kmeans));
    cudaMemcpy(d_kmeans, &h_kmeans, sizeof(Kmeans), cudaMemcpyHostToDevice);

    // Treinamento
    fit(&kmeans, alunos);

    exportarResultados(alunos, numeroAlunos);

    // Visulalizar centroides
    for (int i = 0; i < kmeans.k; i++)
    {
        printf("Centroid %d - Media: %.2f, Numero de Faltas: %.2f\n", i, kmeans.centroids[i].media, kmeans.centroids[i].numeroFaltas);
    }

    // Predição de um novo aluno
    Aluno novoAluno;
    novoAluno.media = 0.95;
    novoAluno.numeroFaltas = 0.95;

    predict(&kmeans, &novoAluno);
    printf("Novo aluno - Media: %.2f, Numero de Faltas: %.2f, Cluster: %d\n", novoAluno.media, novoAluno.numeroFaltas, novoAluno.cluster);

    free(kmeans.centroids);
    free(alunos);

    return 0;
}

UsageError: Cell magic `%%Kmeans` not found.
